In [3]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate, iSwapGate, CPhaseGate
from qiskit.quantum_info import Operator, process_fidelity
from qiskit.synthesis import TwoQubitBasisDecomposer


phi = np.pi / 6

# Target: iSWAP * CP(pi/6)
target = QuantumCircuit(2, name="iswap_cp")
target.append(iSwapGate(), [0, 1])
target.append(CPhaseGate(phi), [0, 1])

U_target = Operator(target).data

# Synthesize the WHOLE 2-qubit unitary into CX basis
decomposer = TwoQubitBasisDecomposer(
    CXGate(),
    euler_basis="U3",
)

synth = decomposer(U_target)

# Clean into explicit u3 + cx gates
synth = transpile(
    synth,
    basis_gates=["u3", "cx"],
    optimization_level=0,
)

print("CX count:", synth.decompose(reps=20).count_ops().get("cx", 0))
print("Process fidelity:", process_fidelity(Operator(synth), Operator(target)))

# Text version
print(synth.draw("text"))

# Matplotlib plot
fig = synth.draw(
    output="mpl",
    fold=-1,
    idle_wires=False,
)

# fig.savefig("iswap_cp_synthesized_circuit.png", dpi=300, bbox_inches="tight")
plt.show()

CX count: 3
Process fidelity: 1.0000000000000004
global phase: 3.963
       ┌──────────────────┐            ┌──────────────────┐          »
q_0: ──┤ U3(π/2,π/2,-π/4) ├────■───────┤ U3(π/2,-π/2,π/2) ├───────■──»
     ┌─┴──────────────────┴─┐┌─┴─┐┌────┴──────────────────┴────┐┌─┴─┐»
q_1: ┤ U3(1.1179,π/2,-3π/4) ├┤ X ├┤ U3(1.3782,0.41245,0.41245) ├┤ X ├»
     └──────────────────────┘└───┘└────────────────────────────┘└───┘»
«          ┌─────────────────┐           ┌──────────────────┐
«q_0: ─────┤ U3(π/12,-π,π/2) ├───────■───┤ U3(π/2,5π/6,π/2) ├
«     ┌────┴─────────────────┴────┐┌─┴─┐┌┴──────────────────┤
«q_1: ┤ U3(1.9752,2.9318,-2.0668) ├┤ X ├┤ U3(π/2,2.1651,-π) ├
«     └───────────────────────────┘└───┘└───────────────────┘
